<a href="https://colab.research.google.com/github/UmymaM/DeepFake-Detection/blob/main/src/train_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")
# mounting drive to access the dataset+model

Mounted at /content/drive


In [2]:
# GRAVEX-200K dataset
import zipfile as zf
with zf.ZipFile("/content/drive/MyDrive/archive (1).zip","r") as zip_ref:
  zip_ref.extractall("/content")

In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from torchvision.models import EfficientNet_B4_Weights
from tqdm import tqdm


In [4]:
train_df=pd.read_csv("/content/drive/MyDrive/train_metadata.csv")
test_df=pd.read_csv("/content/drive/MyDrive/test_metadata.csv")
val_df=pd.read_csv("/content/drive/MyDrive/val_metadata.csv")

In [5]:
train_df.drop_duplicates(inplace=True)
test_df.drop_duplicates(inplace=True)
val_df.drop_duplicates(inplace=True)

In [6]:
train_df.head()

,image_path,label
0,/content/my_real_vs_ai_dataset/my_real_vs_ai_d...,1
1,/content/my_real_vs_ai_dataset/my_real_vs_ai_d...,0
2,/content/my_real_vs_ai_dataset/my_real_vs_ai_d...,1
3,/content/my_real_vs_ai_dataset/my_real_vs_ai_d...,1
4,/content/my_real_vs_ai_dataset/my_real_vs_ai_d...,1


In [7]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

Using Device: cuda


In [8]:
train_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.3,contrast=0.3,saturation=0.3,hue=0.06),
    transforms.GaussianBlur(kernel_size=3,sigma=(0.1,3.0)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
val_transforms=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [9]:
class DeepFakeDataset(Dataset):
  def __init__(self,df,transform=None):
    self.df=df
    self.transform=transform
  def __len__(self):
    return len(self.df)
  def __getitem__(self,idx):
    img_path=self.df.iloc[idx,0]
    label=self.df.iloc[idx,1]
    image=Image.open(img_path).convert("RGB")
    if self.transform:
      image=self.transform(image)
    return image,torch.tensor(label, dtype=torch.float32)

In [10]:
train_dataset=DeepFakeDataset(train_df,train_transform)
val_dataset=DeepFakeDataset(val_df,val_transforms)
test_dataset=DeepFakeDataset(test_df,val_transforms)

In [11]:
train_loader=DataLoader(train_dataset,batch_size=64,shuffle=True,num_workers=2,pin_memory=True)
val_loader=DataLoader(val_dataset,batch_size=64,shuffle=False,num_workers=2,pin_memory=True)
test_loader=DataLoader(test_dataset,batch_size=64,shuffle=False,num_workers=2,pin_memory=True)

In [12]:
images,labels=next(iter(train_loader))
print(f"Batch shape: {images.shape}\nLabels: {labels[:8]}\nLabel dtype: {labels.dtype}")

Batch shape: torch.Size([64, 3, 224, 224])
Labels: tensor([0., 1., 0., 1., 1., 0., 0., 0.])
Label dtype: torch.float32


In [16]:
model=models.efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)

Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:01<00:00, 39.3MB/s]


In [17]:
model.classifier

Sequential(
  (0): Dropout(p=0.4, inplace=True)
  (1): Linear(in_features=1792, out_features=1000, bias=True)
)

In [18]:
num_features=model.classifier[1].in_features
num_features

1792

In [19]:
model.classifier=nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features=num_features, out_features=1, bias=True)
)
model.classifier

Sequential(
  (0): Dropout(p=0.4, inplace=False)
  (1): Linear(in_features=1792, out_features=1, bias=True)
)

In [20]:
# loading saved weights
model.load_state_dict(torch.load("/content/drive/MyDrive/best_model.pth"))

<All keys matched successfully>

In [21]:
model=model.to(device)

In [22]:
criterion=nn.BCEWithLogitsLoss()
optimizer=optim.Adam(model.parameters(),lr=1e-4, weight_decay=1e-2)
scheduler=optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=10)

In [23]:
def train_one_epoch(model, criterion, optimizer, dataloader, device):
    model.train()
    running_loss=0.0
    all_preds=[]
    all_labels=[]
    for images,labels in tqdm(dataloader,desc="Training"):
      images=images.to(device)
      labels=labels.to(device).unsqueeze(1)
      optimizer.zero_grad()
      outputs=model(images)
      loss=criterion(outputs,labels)
      loss.backward()
      optimizer.step()
      running_loss+=loss.item()
      predictions=torch.sigmoid(outputs).round().detach().cpu().numpy()
      all_preds.extend(predictions)
      all_labels.extend(labels.cpu().numpy())
    epoch_loss=running_loss/len(dataloader)
    epoch_accuracy=accuracy_score(all_labels,all_preds)
    return epoch_loss, epoch_accuracy

In [24]:
def validate(model,dataloader,criterion,device):
  model.eval()
  running_loss=0.0
  all_preds=[]
  all_labels=[]
  with torch.no_grad():
    for images,labels in tqdm(dataloader,desc="Validation"):
      images=images.to(device)
      labels=labels.to(device).unsqueeze(1)
      outputs=model(images)
      loss=criterion(outputs,labels)
      running_loss+=loss.item()
      predictions=torch.sigmoid(outputs).round().detach().cpu().numpy()
      all_preds.extend(predictions)
      all_labels.extend(labels.cpu().numpy())
  epoch_loss=running_loss/len(dataloader)
  epoch_accuracy=accuracy_score(all_labels,all_preds)
  return epoch_loss,epoch_accuracy

In [ ]:
epochs=10
best_val_loss = float('inf')
history={'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
for epoch in range(5,epochs):
  print(f"Epoch {epoch+1}/{epochs}")
  train_loss,train_accuracy=train_one_epoch(model,criterion,optimizer,train_loader,device)
  val_loss,val_accuracy=validate(model,val_loader,criterion,device)
  scheduler.step()
  history['train_loss'].append(train_loss)
  history['val_loss'].append(val_loss)
  history['train_acc'].append(train_accuracy)
  history['val_acc'].append(val_accuracy)
  print(f"Train Loss: {train_loss:.5f}  | Train Accuracy: {train_accuracy:.5f}")
  print(f"Val Loss: {val_loss:.5f}  | Val Accuracy: {val_accuracy:.5f}")
  if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(),"/content/drive/MyDrive/best_model.pth")
        print("Saved the best model =^.^=")

Epoch 6/10


Validation: 100%|██████████| 469/469 [02:08<00:00,  3.65it/s]


Train Loss: 0.32057  | Train Accuracy: 0.84717
Val Loss: 0.33439  | Val Accuracy: 0.84127
Saved the best model =^.^=
Epoch 7/10


Validation: 100%|██████████| 469/469 [02:00<00:00,  3.88it/s]


Train Loss: 0.30886  | Train Accuracy: 0.85254
Val Loss: 0.24444  | Val Accuracy: 0.88757
Saved the best model =^.^=
Epoch 8/10


Validation: 100%|██████████| 469/469 [01:59<00:00,  3.92it/s]


Train Loss: 0.28622  | Train Accuracy: 0.86357
Val Loss: 0.22211  | Val Accuracy: 0.88983
Saved the best model =^.^=
Epoch 9/10


Training:  87%|████████▋ | 1909/2188 [29:00<05:04,  1.09s/it]